# 2. Build and run a query

Queries are built from one or more **sub-queries** (`picsure::createSubQuery()`) which can stand alone or be combined into a group with `picsure::buildQuery()`. `picsure::runQuery()` executes them.

This notebook builds a single categorical FILTER from a dictionary search result and asks the backend for a participant count.

In [ ]:
library(picsure)

In [ ]:
token_file <- "token.txt"
my_token <- readLines(token_file, warn = FALSE)[1]

In [ ]:
authorized_session <- picsure::connect(
  platform = picsure::Platform$BDC_AUTHORIZED,
  token    = my_token
)

## Find a variable inside a single study

In [ ]:
framingham_facet <- picsure::facets(authorized_session)
picsure::addFacet(framingham_facet, "dataset_id", "tutorial-biolincc_framingham")

In [ ]:
tutorial_search_results <- picsure::searchDictionary(
  authorized_session,
  "Current cigarette smoking at exam",
  facets = framingham_facet
)
tutorial_search_results$values

In [ ]:
cursmoke_var <- tutorial_search_results[tutorial_search_results$name == "CURSMOKE", ]
cursmoke_var

In [ ]:
raw_values <- cursmoke_var$values[[1]]
raw_values

## Build a categorical FILTER

Pass the concept path and the categorical values to match. `categories` accepts a vector or list.

In [ ]:
cursmoke <- picsure::createSubQuery(
  cursmoke_var$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = raw_values
)

## Run for a count

`runQuery()` accepts a single sub-query or a group built with `buildQuery()`. `type = picsure::QueryType$COUNT` returns a `CountResult` with `$value`, `$margin`, and `$cap` slots; `$value` is `NULL` for small-cohort obfuscation.

In [ ]:
count <- picsure::runQuery(authorized_session, cursmoke, type = picsure::QueryType$COUNT)
count$value